# DP/SFA SSN accuracy in the three-phase domain

`DP_SSN_RLC_accuracy` showed that the DP-SSN accuracy advantage over EMT-SSN
peaks at the carrier and falls off with the signal-to-carrier offset, using a
single-phase series RLC one-port. The `dp.ph3` SSN components apply that same
per-phase real-augmented recurrence with diagonal (uncoupled) R, L, C
matrices, so each phase reproduces the single-phase result exactly; this
notebook confirms that on the phase A envelope of the same balanced series
RLC one-port (R = 10 Ohm, L = 0.01 H, C = 0.002 F) used in
`DP_generalizedSSN_RLC`, using a small-step EMT reference as the truth.

Same three sweeps as the single-phase notebook: time step at the carrier,
signal frequency at a fine time step, and signal frequency at a large time
step.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import dpsimpy
from villas.dataprocessing.readtools import read_timeseries_csv

In [ ]:
param = np.eye(3)

R = 10.0
L = 0.01
C = 0.002

f0 = 50.0
vref = dpsimpy.Math.single_phase_variable_to_three_phase(
    1.0 * np.exp(1j * np.deg2rad(-90.0))
)

dt_ref = 1e-5
tf = 0.3

log_dir = "logs"

In [ ]:
def run(sim_name, system, domain, comp, attr, dt):
    dpsimpy.Logger.set_log_dir(f"{log_dir}/{sim_name}")
    logger = dpsimpy.Logger(sim_name)
    logger.log_attribute("y", attr, comp)

    sim = dpsimpy.Simulation(sim_name, dpsimpy.LogLevel.off)
    sim.set_system(system)
    sim.set_domain(domain)
    sim.set_time_step(dt)
    sim.set_final_time(tf)
    sim.add_logger(logger)
    sim.run()

    return read_timeseries_csv(
        f"{log_dir}/{sim_name}/{sim_name}.csv", print_status=False
    )["y_0"]


def steady_phasor(series, w):
    # Steady-state phasor via least-squares fit to cos/sin at w: the metric
    # that exposes frequency warping (a raw-waveform RMSE hides it at large
    # steps).
    t, y = series.time, series.values
    half = slice(len(t) // 2, None)
    ts, ys = t[half], y[half]
    basis = np.column_stack([np.cos(w * ts), np.sin(w * ts), np.ones_like(ts)])
    a, b, _ = np.linalg.lstsq(basis, ys, rcond=None)[0]
    return a - 1j * b


def dp_node(name):
    return dpsimpy.dp.SimNode(name, dpsimpy.PhaseType.ABC)


def emt_node(name):
    return dpsimpy.emt.SimNode(name, dpsimpy.PhaseType.ABC)

In [ ]:
def build_ssn(ph3, node, gnd, src_freq):
    n1 = node("n1")
    vs = ph3.VoltageSource("vs")
    vs.set_parameters(vref, src_freq)
    rlc = ph3.Full_Serial_RLC("rlc")
    rlc.set_parameters(R * param, L * param, C * param)
    vs.connect([gnd, n1])
    rlc.connect([n1, gnd])
    return dpsimpy.SystemTopology(f0, [n1], [vs, rlc]), rlc


def build_rlc(ph3, node, gnd, src_freq):
    n1, n2, n3 = node("n1"), node("n2"), node("n3")
    vs = ph3.VoltageSource("vs")
    vs.set_parameters(vref, src_freq)
    res = ph3.Resistor("r")
    res.set_parameters(R * param)
    ind = ph3.Inductor("l")
    ind.set_parameters(L * param)
    cap = ph3.Capacitor("c")
    cap.set_parameters(C * param)
    vs.connect([gnd, n1])
    res.connect([n1, n2])
    ind.connect([n2, n3])
    cap.connect([n3, gnd])
    return dpsimpy.SystemTopology(f0, [n1, n2, n3], [vs, res, ind, cap]), ind

`reference_phasor` runs a fine-step EMT simulation and returns the phase A
steady-state phasor at the signal frequency (the truth); `emt_ssn_error` and
`dp_ssn_error` run the SSN models at a given step and signal frequency and
return the phase A phasor error against that reference. The DP source
frequency is the offset from the carrier; the DP result is reconstructed with
`frequency_shift` before the phasor is extracted.

`EMT::Ph3::VoltageSource` scales its reference by `RMS3PH_TO_PEAK1PH`
(`sqrt(2/3)`) before stamping it, while `DP::Ph3::VoltageSource` uses the
reference directly as the envelope amplitude, so the EMT-derived reference
phasor is divided by that factor before comparing it to the DP-SSN phasor.
Skipping this correction adds a constant offset to `dp_ssn_error` that looks
like a discretization error but is pure unit convention.

In [ ]:
dp_gnd = dpsimpy.dp.SimNode.gnd
emt_gnd = dpsimpy.emt.SimNode.gnd


def reference_phasor(f_sig):
    sys, comp = build_rlc(dpsimpy.emt.ph3, emt_node, emt_gnd, f_sig)
    series = run(f"REF_{f_sig:g}Hz", sys, dpsimpy.Domain.EMT, comp, "i_intf", dt_ref)
    return steady_phasor(series, 2.0 * np.pi * f_sig)


def emt_ssn_error(f_sig, dt, ref):
    sys, comp = build_ssn(dpsimpy.emt.ph3, emt_node, emt_gnd, f_sig)
    series = run(
        f"EMT_Ph3_SSN_{f_sig:g}Hz_{dt:.0e}", sys, dpsimpy.Domain.EMT, comp, "i_intf", dt
    )
    return np.abs(steady_phasor(series, 2.0 * np.pi * f_sig) - ref)


def dp_ssn_error(f_sig, dt, ref_dp):
    sys, comp = build_ssn(dpsimpy.dp.ph3, dp_node, dp_gnd, f_sig - f0)
    series = run(
        f"DP_Ph3_SSN_{f_sig:g}Hz_{dt:.0e}", sys, dpsimpy.Domain.DP, comp, "i_intf", dt
    )
    shifted = series.frequency_shift(f0)
    return np.abs(steady_phasor(shifted, 2.0 * np.pi * f_sig) - ref_dp)

## Sweep over time step, source on the carrier

Trapezoidal integration warps the L and C companion models by
`Psi(w) = tan(w h / 2) / (w h / 2)`. With the source on the carrier the
EMT-SSN 50 Hz forced response sits where `Psi > 1`, so its error grows with
the step `h`; the DP-SSN forced response is the DC of the envelope
(`Psi ~ 1`), so it stays flat, exactly as in the single-phase case.

In [ ]:
ref_carrier = reference_phasor(f0)
ref_carrier_dp = ref_carrier / dpsimpy.RMS3PH_TO_PEAK1PH
steps = [5e-3, 2e-3, 1e-3, 5e-4, 2e-4, 1e-4]

err_emt = [emt_ssn_error(f0, dt, ref_carrier) for dt in steps]
err_dp = [dp_ssn_error(f0, dt, ref_carrier_dp) for dt in steps]

for dt, ee, ed in zip(steps, err_emt, err_dp):
    print(
        f"dt={dt:.0e}  EMT-SSN err={ee:.3e}  DP-SSN err={ed:.3e}  ratio={ee / ed:.0f}"
    )

In [ ]:
plt.figure(figsize=(9, 4))
plt.loglog(steps, err_emt, "o-", label="EMT-SSN")
plt.loglog(steps, err_dp, "s-", label="DP-SSN")
plt.xlabel("time step [s]")
plt.ylabel("phase A steady-state phasor error [A]")
plt.title("Accuracy versus time step (source on the 50 Hz carrier), dp.ph3 vs emt.ph3")
plt.legend()
plt.grid(True, which="both", linestyle=":")
plt.show()

In [ ]:
assert err_dp[0] < err_emt[0] / 100
assert max(err_dp) < 20 * min(err_dp)

## Sweep over signal frequency (fine time step)

The source is detuned from the carrier and swept on both sides of it, same
symmetric band as the single-phase notebook. The error is reported relative
to the reference phasor, since the RLC response amplitude varies by orders of
magnitude across the band. At this fine step both methods sit near the floor
set by the reference discretisation; DP-SSN is most accurate at the carrier,
and below roughly 25-30 Hz EMT-SSN wins (the same crossover as in the
single-phase case).

In [ ]:
freqs = [10, 20, 25, 30, 35, 40, 45, 50, 55, 60, 70, 90, 120, 160]
refs = [reference_phasor(f) for f in freqs]
refs_dp = [r / dpsimpy.RMS3PH_TO_PEAK1PH for r in refs]
carrier = freqs.index(50)

dt_fine = 1e-4
emt_fine = [emt_ssn_error(f, dt_fine, r) / abs(r) for f, r in zip(freqs, refs)]
dp_fine = [
    dp_ssn_error(f, dt_fine, r_dp) / abs(r) for f, r_dp, r in zip(freqs, refs_dp, refs)
]

for f, ee, ed in zip(freqs, emt_fine, dp_fine):
    print(
        f"f_sig={f:3d} Hz (offset {f - f0:+.0f})  EMT-SSN rel={ee:.3e}  DP-SSN rel={ed:.3e}  ratio={ee / ed:.2f}"
    )

In [ ]:
plt.figure(figsize=(9, 4))
plt.semilogy(freqs, emt_fine, "o-", label="EMT-SSN")
plt.semilogy(freqs, dp_fine, "s-", label="DP-SSN")
plt.axvline(f0, color="0.7", linestyle=":", label="carrier")
plt.xlabel("signal frequency [Hz]")
plt.ylabel("relative phasor error")
plt.title(
    "Accuracy versus signal frequency (fine step, dt = 1e-4 s), dp.ph3 vs emt.ph3"
)
plt.legend()
plt.grid(True, which="both", linestyle=":")
plt.show()

In [ ]:
assert dp_fine[carrier] < emt_fine[carrier]
assert dp_fine[0] > emt_fine[0]

## Sweep over signal frequency (large time step)

The same symmetric sweep at a large step clears the floor and makes the
effect dramatic, matching the single-phase result.

In [ ]:
dt_big = 2e-3
emt_big = [emt_ssn_error(f, dt_big, r) / abs(r) for f, r in zip(freqs, refs)]
dp_big = [
    dp_ssn_error(f, dt_big, r_dp) / abs(r) for f, r_dp, r in zip(freqs, refs_dp, refs)
]

for f, ee, ed in zip(freqs, emt_big, dp_big):
    print(
        f"f_sig={f:3d} Hz (offset {f - f0:+.0f})  EMT-SSN rel={ee:.3e}  DP-SSN rel={ed:.3e}  ratio={ee / ed:.2f}"
    )

In [ ]:
plt.figure(figsize=(9, 4))
plt.semilogy(freqs, emt_big, "o-", label="EMT-SSN")
plt.semilogy(freqs, dp_big, "s-", label="DP-SSN")
plt.axvline(f0, color="0.7", linestyle=":", label="carrier")
plt.xlabel("signal frequency [Hz]")
plt.ylabel("relative phasor error")
plt.title(
    "Accuracy versus signal frequency (large step, dt = 2e-3 s), dp.ph3 vs emt.ph3"
)
plt.legend()
plt.grid(True, which="both", linestyle=":")
plt.show()

In [ ]:
advantage = [ee / ed for ee, ed in zip(emt_big, dp_big)]
print("DP-SSN advantage at the carrier:", advantage[carrier], "x")
print("DP-SSN advantage far below the carrier:", advantage[0], "x")
assert advantage[carrier] > 100
assert advantage[0] < 1